# **ENSO Growth/Decay Rate Experiment Split by Eruption Location - Reconstructions**

Runs Both: 

(1) Phase-conditioned (eruption-year phase = El Niño / Neutral / La Niña)

Output: 

-  CSV saved to ~/ENSO-GDR-Recons-Location.csv

In [1]:
import numpy as np
import pandas as pd


# SETTINGS

file_path = "/home/563/ft3359/ENSO_Records_all.csv"
YEAR_COL = "Years"

RECON_COLUMNS = [
    "Zhu et al. (2022) Li13b6.",
    "Wilson et al. (2010) Nino34 COAPCR",
    "Freund et al. (2019) Nino4 DJF",
    "Li et al. (2011) NADA PC1",
    "Stahle et al. (1993)",
    "DArrigo et al. (2005) Nino3",
    "Datwyler et al. (2020) ENSO DJF",
    "Geay et al. (2013) Nino3",
    "Liu et al. (2024) PCR",
]

PRE_YEARS = 5
POST_YEARS = 5
FIT_START = 0
FIT_END = 3
THRESHOLD_RAW = 0.5

EXCLUDE_MARGIN_YEARS = 5
N_MC = 6000
SEED = 42
MIN_VALID_EVENTS = 2
LOG_TOL = 1e-6

TROPICAL_LAT_BOUND = 23.0
LOCATIONS = ["Tropical", "NH Extratropical", "SH Extratropical"]
PHASES = ["El Niño", "Neutral", "La Niña"]

# ERUPTION LIST: all 20 eruptions (850-1849 CE) with known seasonality/location.

ERUPTIONS_RAW = [
    (939.0,  4.0,  1.0,  63.6, -1.0, 16.23, 4.97),
    (946.0, 11.0,  1.0,  42.0, -1.0,  1.72, 0.61),
    (1257.0, 7.0,  1.0,  -8.4,  1.4, 59.42, 10.86),
    (1477.0, 2.0,  1.0,  64.6, -1.0,  5.12, 1.61),
    (1510.0, 7.0, 25.0,  64.0, -1.0,  2.30, 0.83),
    (1585.0, 1.0, 10.0,  19.5, 10.6,  8.51, 2.34),
    (1595.0, 3.0,  1.0,   4.9,  0.8,  8.87, 1.51),
    (1600.0, 2.0, 17.0, -16.6,  2.0, 18.95, 4.03),
    (1640.0, 12.0, 26.0,  6.1,  2.8, 18.68, 4.28),
    (1667.0, 9.0, 23.0,  42.7, -1.0,  3.48, 1.11),
    (1673.0, 5.0, 20.0,   1.4,  0.7,  4.67, 0.82),
    (1707.0, 12.0, 16.0,  35.4, -1.0,  1.08, 0.40),
    (1721.0, 5.0, 11.0,  63.6, -1.0,  0.81, 0.36),
    (1739.0, 8.0, 19.0,  42.7, -1.0,  3.44, 1.09),
    (1755.0, 10.0, 17.0,  63.6, -1.0,  1.18, 0.43),
    (1766.0, 4.0,  5.0,  64.0, -1.0,  2.52, 0.75),
    (1783.0, 6.0, 15.0,  64.4, -1.0, 20.81, 7.04),
    (1815.0, 4.0, 10.0,  -8.0,  0.8, 28.08, 4.49),
    (1822.0, 10.0,  8.0,  -7.3, 10.0,  2.02, 0.79),
    (1835.0, 1.0, 20.0,  13.0,  2.0,  9.48, 2.21),
]
eruptions_df = pd.DataFrame(
    ERUPTIONS_RAW,
    columns=["yearCE", "month", "day", "lat", "hemi", "ssi", "sigma_ssi"],
)

eruption_years = eruptions_df["yearCE"].to_numpy(dtype=int)

def classify_location(lat: float) -> str:
    if abs(lat) <= TROPICAL_LAT_BOUND:
        return "Tropical"
    return "NH Extratropical" if lat > TROPICAL_LAT_BOUND else "SH Extratropical"

eruptions_df["location"] = eruptions_df["lat"].apply(classify_location)
LOCATION_BY_YEAR = dict(zip(eruptions_df["yearCE"].astype(int), eruptions_df["location"]))

# LOAD DATA
df = pd.read_csv(file_path)
df.columns = (
    df.columns.astype(str)
    .str.replace(r"^'+|'+$", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

if YEAR_COL not in df.columns:
    raise KeyError(f"'{YEAR_COL}' not found. Found:\n{list(df.columns)}")

df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors="coerce")
df = df.dropna(subset=[YEAR_COL]).copy()
df[YEAR_COL] = df[YEAR_COL].astype(int)
df = df.set_index(YEAR_COL).sort_index()

for c in df.columns:
    df[c] = pd.to_numeric(df[c], errors="coerce")

missing = [c for c in RECON_COLUMNS if c not in df.columns]
if missing:
    raise KeyError(
        "Requested reconstruction columns not found:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# HELPERS

def classify_years_all(series_raw, threshold=0.5):
    el, neu, la = [], [], []
    for y, v in series_raw.items():
        if not np.isfinite(v):
            continue
        if v >= threshold:
            el.append(int(y))
        elif v <= -threshold:
            la.append(int(y))
        else:
            neu.append(int(y))
    return np.array(el, dtype=int), np.array(neu, dtype=int), np.array(la, dtype=int)


def build_all_windows(series_sd, pre_years=5, post_years=5):
    years = series_sd.index.values.astype(int)
    vals = series_sd.values.astype(float)
    year_to_i = {int(y): i for i, y in enumerate(years)}

    lags = np.arange(-pre_years, post_years + 1, dtype=int)
    windows = []
    years_win = []

    for center in years:
        center = int(center)
        w = []
        ok = True
        for lag in lags:
            yy = center + int(lag)
            if yy not in year_to_i:
                ok = False
                break
            v = vals[year_to_i[yy]]
            if not np.isfinite(v):
                ok = False
                break
            w.append(v)
        if ok:
            windows.append(w)
            years_win.append(center)

    if len(windows) == 0:
        return lags, None, None

    return lags, np.asarray(windows, dtype=float), np.asarray(years_win, dtype=int)


def get_window_row(windows, years_win, year):
    idx = np.where(years_win == int(year))[0]
    if idx.size == 0:
        return None
    return windows[idx[0]]


def fit_exponential_rate(comp, pre_years=5, fit_start=0, fit_end=3, tol=1e-6):
    if comp is None:
        return np.nan, np.nan

    idx0 = pre_years + fit_start
    idx1 = pre_years + fit_end + 1
    y = np.asarray(comp[idx0:idx1], dtype=float)

    if np.sum(np.isfinite(y)) < 3:
        return np.nan, np.nan

    tail = y[-2:]
    if np.sum(np.isfinite(tail)) == 0:
        return np.nan, np.nan
    C = np.nanmedian(tail)

    dev = np.abs(y - C)
    valid = np.isfinite(dev) & (dev > tol)

    if np.sum(valid) < 3:
        return np.nan, np.nan

    t = np.arange(fit_start, fit_end + 1, dtype=float)[valid]
    log_dev = np.log(dev[valid])

    try:
        r, alpha = np.polyfit(t, log_dev, 1)
    except Exception:
        return np.nan, np.nan

    tau = -1.0 / r if np.isfinite(r) and (r < 0) else np.nan
    return float(r), float(tau) if np.isfinite(tau) else np.nan


def make_control_pool_mask(years_win, eruption_years, exclude_margin_years=5):
    ok = np.ones_like(years_win, dtype=bool)
    for ey in eruption_years:
        ey = int(ey)
        ok &= ~((years_win >= ey - exclude_margin_years) & (years_win <= ey + exclude_margin_years))
    return ok


def mc_pvalue_for_case(windows, years_win, observed_years, control_pool_years,
                        n_mc, seed, pre_years, fit_start, fit_end, tol):
    observed_years = np.asarray(observed_years, dtype=int)
    control_pool_years = np.asarray(control_pool_years, dtype=int)

    if observed_years.size < MIN_VALID_EVENTS:
        return np.nan, np.nan, np.nan

    obs_windows = [get_window_row(windows, years_win, y) for y in observed_years]
    obs_windows = [w for w in obs_windows if w is not None]
    if len(obs_windows) < MIN_VALID_EVENTS:
        return np.nan, np.nan, np.nan

    obs_comp = np.mean(np.asarray(obs_windows, dtype=float), axis=0)
    r_obs, tau_obs = fit_exponential_rate(obs_comp, pre_years, fit_start, fit_end, tol)

    if not np.isfinite(r_obs):
        return np.nan, np.nan, np.nan

    years_win_set = set(int(y) for y in years_win)
    control_candidates = np.unique(
        [int(y) for y in control_pool_years if int(y) in years_win_set]
    )

    n_events = len(obs_windows)
    if control_candidates.size < n_events:
        return r_obs, tau_obs, np.nan

    rng = np.random.default_rng(seed)
    r_null = []

    for _ in range(n_mc):
        draw = rng.choice(control_candidates, size=n_events, replace=False)
        draw_windows = [get_window_row(windows, years_win, y) for y in draw]
        comp_mc = np.mean(np.asarray(draw_windows, dtype=float), axis=0)
        r_mc, _ = fit_exponential_rate(comp_mc, pre_years, fit_start, fit_end, tol)
        if np.isfinite(r_mc):
            r_null.append(r_mc)

    if len(r_null) == 0:
        return r_obs, tau_obs, np.nan

    r_null = np.asarray(r_null, dtype=float)
    r_med = np.nanmedian(r_null)
    p = np.mean(np.abs(r_null - r_med) >= np.abs(r_obs - r_med))

    return r_obs, tau_obs, float(p)

# MAIN
rows = []

for recon_name in RECON_COLUMNS:
    print(f"{recon_name}")

    series_raw = df[recon_name].dropna()
    if series_raw.empty:
        print("  -> No data; skipping.")
        continue

    std_raw = float(series_raw.std())
    if (not np.isfinite(std_raw)) or std_raw == 0:
        print("  -> Invalid std; skipping.")
        continue

    mean_raw = float(series_raw.mean())
    series_sd = (series_raw - mean_raw) / std_raw

    lags, windows, years_win = build_all_windows(series_sd, PRE_YEARS, POST_YEARS)
    if windows is None:
        print("  -> No valid windows; skipping.")
        continue

    el_all, neu_all, la_all = classify_years_all(series_raw, THRESHOLD_RAW)
    el_set, neu_set, la_set = set(el_all), set(neu_all), set(la_all)
    phase_sets = {"El Niño": el_set, "Neutral": neu_set, "La Niña": la_set}

    valid_years = set(int(y) for y in years_win)
    eru_valid = np.array([y for y in eruption_years if int(y) in valid_years], dtype=int)

    pool_mask = make_control_pool_mask(years_win, eruption_years, EXCLUDE_MARGIN_YEARS)
    control_years_all = years_win[pool_mask]

    control_years_by_phase = {
        "El Niño": np.array([y for y in control_years_all if y in el_set]),
        "Neutral": np.array([y for y in control_years_all if y in neu_set]),
        "La Niña": np.array([y for y in control_years_all if y in la_set]),
    }

    for location in LOCATIONS:
        for phase in PHASES:
            eru_lp = np.array(
                [y for y in eru_valid
                 if LOCATION_BY_YEAR.get(int(y)) == location and int(y) in phase_sets[phase]],
                dtype=int
            )

            case_name = f"{location} | {phase}"
            n_events = len(eru_lp)

            if n_events < MIN_VALID_EVENTS:
                rows.append({
                    "dataset": recon_name, "location": location, "phase": phase,
                    "case": case_name, "N_events": n_events,
                    "r_yr1": np.nan, "tau_yr": np.nan, "p_value": np.nan
                })
                continue

            seed_case = abs(hash((recon_name, case_name, SEED))) % (2**32)

            r_obs, tau_obs, p_val = mc_pvalue_for_case(
                windows, years_win, eru_lp, control_years_by_phase[phase],
                N_MC, seed_case, PRE_YEARS, FIT_START, FIT_END, LOG_TOL
            )

            rows.append({
                "dataset": recon_name, "location": location, "phase": phase,
                "case": case_name, "N_events": n_events,
                "r_yr1": r_obs, "tau_yr": tau_obs, "p_value": p_val
            })

results_df = pd.DataFrame(rows)

print("\nGrowth/Decay rate results (exponential fit), Location x Phase, per reconstruction")
if not results_df.empty:
    print(results_df[["dataset","location","phase","N_events","r_yr1","tau_yr","p_value"]].to_string(index=False))
else:
    print("No results produced.")

SAVE_CSV = True
OUTPUT_PATH = "/home/563/ft3359/FT-Honours/Honours_Paper/GDR/Location_Analysis/ENSO-GDR-Recons-Location.csv"

if SAVE_CSV and not results_df.empty:
    results_df.to_csv(OUTPUT_PATH, index=False)
    print(f"\nSaved CSV: {OUTPUT_PATH}")

Zhu et al. (2022) Li13b6.
Wilson et al. (2010) Nino34 COAPCR
Freund et al. (2019) Nino4 DJF
Li et al. (2011) NADA PC1
Stahle et al. (1993)
DArrigo et al. (2005) Nino3
Datwyler et al. (2020) ENSO DJF
Geay et al. (2013) Nino3
Liu et al. (2024) PCR

Growth/Decay rate results (exponential fit), Location x Phase, per reconstruction
                           dataset         location   phase  N_events     r_yr1    tau_yr  p_value
         Zhu et al. (2022) Li13b6.         Tropical El Niño         1       NaN       NaN      NaN
         Zhu et al. (2022) Li13b6.         Tropical Neutral         4 -1.218791  0.820485 0.063095
         Zhu et al. (2022) Li13b6.         Tropical La Niña         4 -0.394994  2.531681 0.723798
         Zhu et al. (2022) Li13b6. NH Extratropical El Niño         4 -0.349015  2.865206 0.486631
         Zhu et al. (2022) Li13b6. NH Extratropical Neutral         1       NaN       NaN      NaN
         Zhu et al. (2022) Li13b6. NH Extratropical La Niña         4 -0.5281